## Setup session
Provide an experiment name as `experiment_name`. Enter a patient identifier as `patient_id`. Provide the current medication state as `medication_state` using either `Off` or `On`. Further details on medication, especially when deviating from the study protocol, must be noted in the electronic labbook. Enter a number as `session_id` to uniquely identify this session in case multiple sessions will be run with the same patient and the same medication condition. This cell also initializes a folder structure for this session. A root directory will be created in `C:\\Measurements\\experiment_name` with subfolders following the BIDS specification. Calibration data recorded via the TMSi GUI and experimental data recorded via Labrecorder will be stored in the sourcedata folder. Folders will only be created if they were not created before.

In [ ]:
from setup.Session import Session

experiment_name = "adbs_stimdur"
patient_id = "test"
medication_state = "Off" # "Off" or "On"
session_id = 1

session = Session(experiment_name, patient_id, medication_state, session_id)

## Set device configuration
Update TMSi Saga configuration: Set channel names, enabled channels and sampling frequency according to config file. Not that AUX and BIP channel names can not be set to individual channel names.

In [ ]:
session.set_saga_configuration()

## Get calibration data
Set `calibration_run_index` to identify calibration data. In case multiple calibration runs need to be executed (e.g., due of artefacts), these can be distinguished by this index.

In [ ]:
calibration_run_index = 1

Record calibration data. In case calibration data with this `calibration_run_index` already exists you will be asked whether you want to record to a file with the same `calibration_run_index`. If not, you need to change `calibration_run_index` to a different value. Calibration data will be saved to the disk and can be loaded in later.

In [ ]:
session.record_calibration_data(calibration_run_index)

Visualize calibration data. First, bipolar derivations will be computed from the channels present in the calibration data according to `config_session.json`. A power spectrum and a spectrogram will be computed and plotted for these channels.

In [ ]:
%matplotlib qt
session.compute_spectra(calibration_run_index)

## Finalize configuration
Select the recording channel that will be used to extract the aDBS biomarker as `adbs_channel`. The stimulation channel will be automatically set according to the recording channel (i.e., the channel in between the cathode and the anode: "sandwich configuration"). Provide the maximum stimulation amplitude to which stimulation will be ramped up as `max_stim_amp`.

In [ ]:
adbs_channel_anode = "X-1" # "LFPR1", "LFPR234", "LFPL1", "LFPL234"
adbs_channel_cathode = "Y-1" # "LFPR567", "LFPR8", "LFPL567", "LFPL8"
max_stim_amp = 2

Load experiment and condition configurations. Add `adbs_channel` and `max_stim_amp` to the configuration. Compute and add thresholds based on calibration data and condition-specific parameters for biomarker calculation (e.g., smoothing, windowing, etc.). With this, a final configuration that is specific for the experiment, patient, session and condition is created. This configuration will be provided to timeflux. 

In [ ]:
session_setup.finalize_configuration(calibration_run_index, adbs_channel_anode, adbs_channel_cathode, max_stim_amp)